# 03 - Is an object read with retp the same as one read with energyml-rust?

Both bridges hand an object over as a dictionary in the EnergyML JSON encoding, `$type` on every node. So an object read from a server by `retp` should go into an `energyml.rust` package, and come out unchanged in what identifies it. And the object read from the server should say the same as the same object read from the EPC that was pushed.

This notebook only reads from the server. Part B reads what notebook 02 pushed, so it needs `RETP_WRITE_DATASPACE` and the EPC; it does not push.

## Before you run

Settings come from environment variables only. No server name, user or token is written in this notebook, and the token is never printed.

| Variable | Needed | Meaning |
|---|---|---|
| `RETP_URL` | yes | `wss://...` ETP endpoint |
| `RETP_TOKEN` | yes | bearer token, without its scheme |
| `RETP_INSECURE` | no | `1` skips the certificate check (default: checked) |
| `RETP_PARTITION` | no | OSDU data partition |
| `RETP_DATASPACE` | no | the dataspace to read; when unset, the first one listed is taken and the notebook says so |
| `RETP_WRITE_DATASPACE` | no | a dataspace you created for the test and will delete; only the writing notebook uses it, and it does nothing without it |
| `RETP_EPC` | no | path of an EPC to push (its `.h5` beside it) |

When a precondition is missing, a cell prints why and the rest of the notebook is skipped without an error. When the server answers something the ETP 1.2 specification forbids, an `assert` fails with a message. The assertions check the **shape** of the answers, never the values a given server holds.

In [ ]:
import retp
import _server

try:
    from energyml.rust.epc import Epc
except ImportError as why:
    Epc = None
    print(f"energyml.rust is not importable ({why}); install it with `pip install energyml-rust`. Nothing to run.")

KW = _server.connect_arguments() if Epc else None
READY = KW is not None

In [ ]:
import re

UUID = re.compile(r"[0-9a-fA-F]{8}-(?:[0-9a-fA-F]{4}-){3}[0-9a-fA-F]{12}")
TYPE = re.compile(r"/([a-z]+[0-9]+)\.([A-Za-z0-9_]+)\(")


def uuid_of(uri):
    found = UUID.findall(uri)
    return found[-1].lower() if found else None


def family(uri):
    found = TYPE.search(uri)
    return found.group(1) if found else None


def qualified_type(uri):
    found = TYPE.search(uri)
    return f"{found.group(1)}.{found.group(2)}" if found else None


def refused(call, *expected):
    # Run call(); it must raise one of expected. Anything else is a failure.
    try:
        call()
    except expected as error:
        return error
    except Exception as other:
        raise AssertionError(f"expected {[e.__name__ for e in expected]}, got {type(other).__name__}: {other}")
    raise AssertionError("the call returned instead of refusing")

In [ ]:
def field(node, name):
    # The generated model's key spelling is not assumed: look it up without the case.
    return next((value for key, value in node.items() if key.lower() == name.lower()), None)


def title(obj):
    citation = field(obj, "Citation")
    return field(citation, "Title") if isinstance(citation, dict) else None


def identity(obj):
    return obj.get("$type"), field(obj, "Uuid"), title(obj)

## Part A - the round trip

The dictionaries read by `retp` are put into an empty in-memory package of `energyml.rust` (`Epc.create`, `put`) and read back (`get`). `$type`, uuid and title must survive. If the bridge refuses an object, the reason is printed, and at least one object must go through.

In [ ]:
if READY:
    with retp.connect(**KW) as session:
        DATASPACE = _server.dataspace_to_read(session)
        uris = [row["uri"] for row in session.resources(DATASPACE)["resources"]] if DATASPACE is not None else []
        objects, errors = session.get_objects(uris[:5]) if uris else ({}, {})
    assert not errors, f"objects listed by the server cannot be read: {errors}"

    if not objects:
        print("The dataspace holds no object: Part A is skipped.")
    else:
        package = Epc.create("roundtrip")
        through = 0
        for uri, obj in objects.items():
            try:
                written = package.put(obj)
            except Exception as refusal:
                print(f"energyml.rust refused {obj['$type']}: {type(refusal).__name__}: {refusal}")
                continue
            again = package.get(written)
            assert identity(again) == identity(obj), f"identity changed: {identity(obj)} became {identity(again)}"
            differing = sorted(key for key in set(obj) | set(again) if obj.get(key) != again.get(key))
            print(obj["$type"], "-", "identical" if not differing else f"same identity; differs in {differing}")
            through += 1
        assert through >= 1, "energyml.rust took none of the objects read by retp"

## Part B - the server against the EPC

The same object, read from the server (`retp`) and from the EPC that was pushed (`energyml.rust`), compared on `$type`, `Uuid` and `Citation.Title`. The objects are matched by uuid; the EPC is opened by `energyml.rust` and the server is read, never written.

In [ ]:
WRITE_DATASPACE = _server.write_dataspace() if READY else None
EPC = _server.epc_path() if WRITE_DATASPACE else None
if EPC is not None:
    with retp.connect(**KW) as session:
        listed = session.resources(WRITE_DATASPACE)["resources"]
        by_uuid = {uuid_of(row["uri"]): row["uri"] for row in listed}
        package = Epc.open(str(EPC))
        summaries = package.objects()
        common = [s for s in summaries if s.uuid.lower() in by_uuid]
        if not common:
            print("None of the EPC's objects is in the dataspace yet: run notebook 02 first.")
        else:
            served, errors = session.get_objects([by_uuid[s.uuid.lower()] for s in common])
            assert not errors, f"objects listed by the server cannot be read: {errors}"
            for summary in common:
                from_server = served[by_uuid[summary.uuid.lower()]]
                from_epc = package.get(summary.uri)
                assert from_server["$type"] == from_epc["$type"], f"$type: {from_server['$type']} against {from_epc['$type']}"
                assert str(field(from_server, "Uuid")).lower() == summary.uuid.lower() == str(field(from_epc, "Uuid")).lower(), f"Uuid differs for {summary.uuid}"
                assert title(from_server) == title(from_epc), f"Citation.Title differs for {summary.uuid}: {title(from_server)!r} against {title(from_epc)!r}"
            print(f"{len(common)} of {len(summaries)} object(s) agree on $type, Uuid and Citation.Title")
print("done")